# Iteration 9: MFE vs MFCC vs linear spectrogram (report entry 090)

Real data only. Iteration 5's best recipe, unchanged except for what the network sees:
pooled rigs, leave one unit out (Nightjar, d3set and the three FPV campaigns, one at a time),
`simdist`, speed perturbation 0.8-1.25, DS-CNN, 1 s window, 200,000 training windows, three seeds.

| Frontend | What the CNN sees |
|---|---|
| `mfe` | 64 log-mel bands, 25 ms frame, 10 ms stride (model 6's frontend) |
| `mfcc` | first 20 MFCC of that log-mel |
| `linear` | log power of every STFT bin 50-8000 Hz, 64 ms frame (15.6 Hz bins), 10 ms stride |

`comb` and `lgbm` run in the `mfe` arm only: they are the fixed baselines.

**Upload before running** (keep the folder layout of the repo):
- `data/processed/`: `manifest.csv`, `range_track.csv`, `summary.json`, `ddl_clips.csv` and the stores
  `nightjar.i16`, `d3set.i16`, `fpv.i16`, `halmstad.i16`, `ddl.f32`, `dronenoise.f32`, `lund.f32` (about 2.1 GB).
- FSD50K, raw as published, into `data/FSD50K/`: `FSD50K.dev_audio.z01`-`.z05` + `.zip`,
  `FSD50K.eval_audio.z01` + `.zip`, `FSD50K.metadata.zip`, `FSD50K.ground_truth.zip`.
  Skip this if `data/processed/fsd50k.i16` and its manifest rows are already there.

In [ ]:
import collections, csv, json, os, queue, shutil, subprocess, sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

REPO = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
os.chdir(REPO)
# Runs use the project's own environment (`uv sync --group dev --extra dl`), not the kernel's Python.
PY = str(REPO / ".venv" / "bin" / "python") if (REPO / ".venv" / "bin" / "python").exists() else sys.executable
DEVICE = "cuda"                      # "cpu" if there is no GPU
GPUS = [0, 1, 2]                     # each GPU takes the next run from a queue
SEEDS = [0, 1, 2]
UNITS = ["nightjar", "d3set", "fpv:ariel_25.02", "fpv:palmahim_10.02", "fpv:palmahim_27.05"]
FRONTENDS = ["mfe", "mfcc", "linear"]
WINDOWS = 200_000
RUNS = REPO / "artifacts" / "runs"
print(REPO, "| runs with", PY)

In [ ]:
# Pascal GPUs (TITAN X, sm_61) need a CUDA 12.6 torch build: the cu130 wheel in uv.lock has no sm_61 kernels.
#   uv sync --group dev --extra dl
#   uv pip install --python .venv/bin/python --reinstall-package torch torch --index-url https://download.pytorch.org/whl/cu126
# Afterwards call .venv/bin/python directly (`uv run` would re-sync to the lock).
if PY == sys.executable:
    subprocess.run([PY, "-m", "pip", "install", "-q", "-e", ".[dl]", "pytest"], check=True)
subprocess.run([PY, "-c", "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), "
                "[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]); "
                "print('gpu ok', torch.ones(2, device='cuda').sum().item())"], check=True)

## Data check

In [ ]:
P = REPO / "data" / "processed"
rows = list(csv.DictReader(open(P / "manifest.csv", newline="", encoding="utf-8")))
print(collections.Counter(r["source"] for r in rows))
for f in sorted({r["file"] for r in rows}) + ["range_track.csv"]:
    print(f"{f:<16} {'ok' if (P / f).exists() else 'MISSING'}")
need = {"nightjar", "d3set", "fpv", "ddl", "dronenoise", "lund", "fsd50k"}
print("sources missing:", sorted(need - {r["source"] for r in rows}) or "none")

## FSD50K: extract and convert (only if `fsd50k` is missing above)

In [ ]:
X = REPO / "data" / "extracted" / "FSD50K"
X.mkdir(parents=True, exist_ok=True)
for part in ("dev", "eval"):
    if (X / f"FSD50K.{part}_audio").exists():
        continue
    z = REPO / "data" / "FSD50K" / f"FSD50K.{part}_audio.zip"     # a multi-part zip: .z01 ... + .zip
    if shutil.which("7z"):
        subprocess.run(["7z", "x", "-y", str(z), f"-o{X}"], check=True, stdout=subprocess.DEVNULL)
    elif shutil.which("zip") and shutil.which("unzip"):
        joined = X / f"_{part}_joined.zip"
        subprocess.run(["zip", "-q", "-s", "0", str(z), "--out", str(joined)], check=True)
        subprocess.run(["unzip", "-q", str(joined), "-d", str(X)], check=True)
        joined.unlink()
    else:
        raise SystemExit("needs 7z or zip+unzip, e.g. conda install -c conda-forge p7zip")
subprocess.run([PY, "-I", "scripts/prepare_data.py", "fsd50k"], check=True)

## Tests

In [ ]:
subprocess.run([PY, "-m", "pytest", "-q", "-p", "no:cacheprovider"], check=True)

## Runs

One run per frontend and seed; each trains and scores all five held-out units (15 trainings per frontend).
The nine runs wait in a queue and each GPU takes the next one; the linear runs go first, as they take longest
(entry 092: the linear network is GPU-bound, the others CPU-bound).
Logs go to `artifacts/runs/<name>.log`; a finished run leaves `<name>.done`, so re-running the cell resumes.

In [ ]:
RUNS.mkdir(parents=True, exist_ok=True)
todo = queue.Queue()
for fe in sorted(FRONTENDS, key=lambda f: f != "linear"):   # the slowest first
    for seed in SEEDS:
        todo.put((fe, seed))

def run(fe, seed, gpu):
    name = f"it9_{fe}_s{seed}"
    if (RUNS / f"{name}.done").exists():
        print("done   ", name, flush=True); return
    models = ["comb", "lgbm", "dscnn"] if fe == "mfe" else ["dscnn"]
    cmd = [PY, "-u", "scripts/run_range.py", "--name", name, "--pool", "--arms", *UNITS,
           "--recipes", "simdist", "--models", *models, "--device", DEVICE, "--windows", str(WINDOWS),
           "--speed", "0.8", "1.25", "--deploy-fa", "0.05", "--seed", str(seed), "--frontend", fe]
    print(f"running {name} on GPU {gpu}", flush=True)
    with open(RUNS / f"{name}.log", "w") as log:
        subprocess.run(cmd, check=True, stdout=log, stderr=subprocess.STDOUT,
                       env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)})
    (RUNS / f"{name}.done").touch()
    print("finished", name, flush=True)

def worker(gpu):
    while True:
        try:
            fe, seed = todo.get_nowait()
        except queue.Empty:
            return
        run(fe, seed, gpu)

with ThreadPoolExecutor(len(GPUS)) as pool:
    for f in [pool.submit(worker, g) for g in GPUS]:
        f.result()

## Summary

Per held-out unit and scorer: recall on all drone windows at 10% and 5% false alarms on that unit's own
no-drone audio (iteration 5's figure is the 10% one), recall within 50 m and at 100-200 m (Nightjar, d3set),
and at the deployment threshold frozen at 5% on validation: recall, false alarms, mower-only false alarms, Lund recall.
Means over seeds; `n` is the number of seeds.

In [ ]:
def g(d, *ks):
    for k in ks:
        d = d.get(k) if isinstance(d, dict) else None
    return d

out = []
for p in sorted(RUNS.glob("it9_*_s*.json")):
    fe, seed = p.stem.split("_")[1], int(p.stem.split("_s")[-1])
    for r in json.loads(p.read_text())["results"]:
        t = r["test"]
        out.append(dict(
            scorer=fe if r["scorer"] not in ("comb (no ML)", "lgbm") else r["scorer"], unit=r["arm"][len("hold out "):], seed=seed,
            recall_10=g(t, "own", "0.10", "by_range", "all", "recall"), recall_05=g(t, "own", "0.05", "by_range", "all", "recall"),
            r0_50_10=g(t, "own", "0.10", "by_range", "0-50 m", "recall"), r100_200_10=g(t, "own", "0.10", "by_range", "100-200 m", "recall"),
            deploy_recall=g(t, "deploy", "by_range", "all", "recall"), deploy_fa=t.get("deploy_fa"),
            mower_fa=g(r, "confuser", "fa_deploy"), lund_recall=g(r, "unseen", "lund", "deploy", "by_range", "all", "recall")))
with open(RUNS / "it9_summary.csv", "w", newline="") as f:
    w = csv.DictWriter(f, out[0].keys()); w.writeheader(); w.writerows(out)

import pandas as pd
df = pd.DataFrame(out)
tab = df.groupby(["unit", "scorer"]).agg(n=("seed", "count"), **{c: (c, "mean") for c in df.columns[3:]}).round(3)
fpv = df[df.unit.str.startswith("fpv")].groupby(["scorer", "seed"]).recall_10.mean().groupby("scorer").agg(["mean", "min", "max"]).round(3)
print("FPV mean recall at 10% own false alarms (iteration 5 MFE reference: 0.655)"); display(fpv)
display(tab)